# VGG-19 learning-rate probe — BaCP magnitude @ 0.95

Does BaCP's uniform LR 0.1 explain VGG-19 underperforming its own I.P. baseline?

**Why two cells, not one.** The existing 89.99 was produced under the previous protocol (60 epochs, delta_T 88, and no validation split on the contrastive recipe). The code now runs 50 epochs, delta_T 87 and a 9:1 split for both recipes, so a single LR-0.05 run would confound the learning rate with the protocol change. Both LRs are therefore run here under identical current code; only `learning_rate` differs.

| | |
|---|---|
| model | vgg19, CIFAR-10, magnitude, 0.95 |
| arms | `learning_rate` 0.10 (current default) vs 0.05 (the original per-model value) |
| records | `.lr0.10` / `.lr0.05` suffixes, so nothing collides with the main table |

**Reading it.** VGG-19 has *zero* BatchNorm layers (ResNet-50 has 53) and 85.3% of its weights sit in the classifier MLP that the features feeding the contrastive head pass through. If 0.05 recovers most of the ~1.0-point gap to I.P. (91.02), the learning rate is the dominant cause and is fixable. If both arms land near 90, the cause is architectural -- the contrastive signal is computed downstream of the most heavily pruned part of the network -- and no LR will fix it.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check

In [ ]:
SEED, GPU = 1, 0
ARMS = [('lr0.10', 0.10), ('lr0.05', 0.05)]

plan = [nb.make_cell('vgg19', 'bacp', seed=SEED, pruner='magnitude',
                     sparsity=0.95, variant=tag, learning_rate=lr)
        for tag, lr in ARMS]
for c in plan:
    print(f"{c['key']:<62} lr={c['config']['learning_rate']}")
assert nb.sanity_check(plan), 'sanity check failed'

## Run — ~20 min each

In [ ]:
for cell in plan:
    nb.run(cell, gpu=GPU)
    nb.update_results_csv()

## Verdict

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
got = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if 'vgg19' in k and 's0.95.magnitude' in k and r.get('status') == 'ok':
        got[k.split('.')[-1] if k.endswith(('lr0.10', 'lr0.05')) else 'old'] = \
            r.get('test_acc_pct')

print(f'{"arm":<26}{"test acc":>10}')
print(f'{"I.P. magnitude 0.95":<26}{91.02:>10.2f}   (old protocol)')
print(f'{"BaCP lr=0.10 (old proto)":<26}{89.99:>10.2f}')
for tag in ('lr0.10', 'lr0.05'):
    v = got.get(tag)
    print(f'{"BaCP " + tag + " (current)":<26}' +
          (f'{v:>10.2f}' if v is not None else f'{"pending":>10}'))
a, b = got.get('lr0.10'), got.get('lr0.05')
if a is not None and b is not None:
    print()
    print(f'lr0.05 - lr0.10 = {b - a:+.2f}')
    print('positive and large -> the learning rate is the dominant cause')
    print('near zero          -> architectural; the contrastive signal sits')
    print('                      downstream of the most-pruned weights')